In [ ]:
import json
import itertools
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr

import analysis_utils
import constants

In [ ]:
agent2color = {'ours_full': 'red', 'ours': 'red', 'expert': 'blue', 'random': 'grey',
               'human': 'pink', 'play': 'pink', 'think': 'pink', 'watch': 'pink'}
model2name = {'ours': 'Intuitive Gamer', 'ours_full': 'Intuitive Gamer', 'expert': 'Expert Gamer',
              'random': 'Random', 'random_terminal': 'Random'}
agents = ['ours', 'expert', 'random']
human_color = 'grey'

In [ ]:
with open('../human-data/play-exp/human-v-human/final-play/final_agg.json', 'r') as f:
    agg_data = json.load(f)

In [ ]:
standard_slider_vals = {0, 50, 100}
all_uids = {}

for stimuli, stimuli_data in agg_data.items():
    for (arena, order2player,
         player_judgements, move_times, surrender_info) in zip(stimuli_data['arena'],
                                                       stimuli_data['order2player'],
                                                       stimuli_data['player_judgements'],
                                                        stimuli_data['move_times'],
                                                        stimuli_data['surrender_info']):
        player_ids = list(player_judgements.keys())
        player2order = {v: k for k,v in order2player.items()}
        for player_id in player_ids:
            all_uids.setdefault(player_id, {'move_times': [], 'surrenders': [],
                                            'immediate_surrenders': [],
                                            'judgements': [],
                                            'arena': None,
                                            'standard_judgements': []})
            all_uids[player_id]['arena'] = arena
        for player_id, judgements in player_judgements.items():
            if np.any([str(s) == 'nan' for s in judgements]): continue
            all_uids[player_id]['judgements'].extend(judgements)
            near_standards = []
            for resp in judgements:
                near_standard = False
                if (resp > 45 and resp < 55) or (resp >= 0 and resp < 10) or (resp > 90):
                    near_standard = True
                near_standards.append(near_standard)
            all_uids[player_id]['standard_judgements'].extend(near_standards)
        if surrender_info is not None:
            player_surrenderd, surr_time, surr_board = surrender_info
            player_id_surrenderd = order2player[str(player_surrenderd)]
            all_uids[player_id_surrenderd]['surrenders'].append(True)
            if np.sum(surr_board) <= 1:
                all_uids[player_id_surrenderd]['immediate_surrenders'].append(True)
        move_time_vals = move_times[1:]
        for player_id, order in player2order.items():
            if len(move_time_vals) == 0: continue
            try:
                if 'The second player can place 2 pieces as their first move' in stimuli:
                    if order == '1':
                        move_times_player = [move_time_vals[0]] + move_time_vals[3::2]
                    elif order == '2':
                        move_times_player = [move_time_vals[1], move_time_vals[2]] + move_time_vals[4::2]
                elif 'The first player can place 2 pieces as their first move' in stimuli:
                    if order == '1':
                        move_times_player = [move_time_vals[0], move_time_vals[1]] + move_time_vals[3::2]
                    elif order == '2':
                        move_times_player = [move_time_vals[2]] + move_time_vals[4::2]
                else:
                    if order == '1':
                        move_times_player = move_time_vals[::2]
                    elif order == '2':
                        move_times_player = move_time_vals[1::2]
                all_uids[player_id]['move_times'].extend(move_times_player)
            except Exception:
                pass

too_many_standard_uids = []
for uid, entry in all_uids.items():
    standard_judgements = entry['standard_judgements']
    prop_standard = np.sum(standard_judgements)/len(standard_judgements)
    if prop_standard > 0.8:
        too_many_standard_uids.append([uid, prop_standard, entry['judgements']])
too_many_standard_uids_set = set([uid for uid, _, _ in too_many_standard_uids])
print('filtered play uids:', len(too_many_standard_uids_set))

In [ ]:
human_play_payoffs = {game: [] for game in agg_data}
human_play_fun = {game: [] for game in agg_data}
game2pdraw = {game: [] for game in agg_data}
game2pwin = {game: [] for game in agg_data}

for game, game_data in agg_data.items():
    all_player_judgements = game_data['player_judgements']
    game_fun_vals = []
    game_payoff_vals = []
    for player_judgements in all_player_judgements:
        for player, judgements in player_judgements.items():
            if player in too_many_standard_uids_set:
                continue
            if len(judgements) == 3:
                pfirst, pdraw, skill = judgements
                if str(pfirst) == 'nan' or str(pdraw) == 'nan':
                    continue
                pwin = analysis_utils.get_pwin(pdraw, pfirst)
                pdraw /= 100
                pwin /= 100
                payoff = analysis_utils.compute_utility(pdraw, pwin)
                game_payoff_vals.append(payoff)
                game2pwin[game].append(pwin)
                game2pdraw[game].append(pdraw)
            elif len(judgements) == 2:
                fun, skill = judgements
                if str(fun) == 'nan': continue
                game_fun_vals.append(fun)
    task_vals = {'fun': game_fun_vals, 'advantage': game_payoff_vals}
    for task, resps in task_vals.items():
        n_resps = len(resps)
        n_rem = int(n_resps * 0.1)
        dists = []
        for idx in range(n_resps):
            resp = resps[idx]
            other_resps = resps[:idx] + resps[idx+1:]
            m = np.mean(other_resps)
            d = np.abs(m - resp)
            dists.append((idx, d))
        sorted_dists = sorted(dists, key=lambda x: x[1], reverse=True)
        rem = [x[0] for x in sorted_dists[:n_rem]]
        new_resps = [r for i, r in enumerate(resps) if i not in rem]
        if task == 'fun':
            human_play_fun[game] = new_resps
        else:
            human_play_payoffs[game] = new_resps

In [ ]:
def switch_player(player, players):
    return [p for p in players if player != p][0]

game2outcomes = {game: [] for game in agg_data}
game2n_matches = {game: 0 for game in agg_data}
game2surrenders = {game: 0 for game in agg_data}
game2surrender_players = {game: [] for game in agg_data}
game2draws = {game: [] for game in agg_data}
human_game_lens = {game: [] for game in agg_data}
fun2game2outcome = {stim: {'fun': [], 'outcome': []} for stim in agg_data}

for stim, stimuli_data in agg_data.items():
    surrenders = [i for i, entry in enumerate(stimuli_data['surrender_info']) if entry is not None]
    n_matches = len(stimuli_data['surrender_info'])
    game2surrenders[stim] = len(surrenders)
    game2n_matches[stim] = n_matches
    for i, entry in enumerate(stimuli_data['draw_info']):
        n_rej, n_accpt = 0, 0
        if len(entry['reject']) != 0:
            n_rej = len(entry['reject'])
        if len(entry['accept']) != 0:
            n_accpt = len(entry['accept'])
        if n_accpt != 0 or n_rej != 0:
            game2draws[stim].append({'idx': i, 'n_rej': n_rej, 'n_accpt': n_accpt})
    for i in surrenders:
        surr_player = stimuli_data['surrender_info'][i][0]
        game2surrender_players[stim].append(surr_player)

    for (arena, outcome, order2player,
         player_judgements, boards, move_times, _, draw_info, surrender_info) in zip(
             stimuli_data['arena'], stimuli_data['outcome'],
             stimuli_data['order2player'],
             stimuli_data['player_judgements'],
             stimuli_data['boards'], stimuli_data['move_times'],
             stimuli_data['outcome'], stimuli_data['draw_info'], stimuli_data['surrender_info']):
        player2order = {v: k for k,v in order2player.items()}
        accepted = len(draw_info['accept']) >= 1
        if outcome == 'Draw':
            game_outcome = 0
        else:
            game_outcome = int(player2order[outcome])
        game2outcomes[stim].append(game_outcome)
        player_vals = list(player_judgements.values())
        if np.any([len(vals) != 2 for vals in player_vals]): continue
        game_len = len(eval(boards))
        if outcome == 'Draw':
            fun_vals = [vals[0] for vals in list(player_judgements.values())]
            fun2game2outcome[stim]['fun'].extend(fun_vals)
            fun2game2outcome[stim]['outcome'].extend(['Draw', 'Draw'])
        else:
            winner_fun = player_judgements[outcome][0]
            loser_fun = player_judgements[switch_player(outcome, player_judgements.keys())][0]
            if str(winner_fun) == 'nan' or str(loser_fun) == 'nan': continue
            fun2game2outcome[stim]['fun'].append(winner_fun)
            fun2game2outcome[stim]['outcome'].append('Won')
            fun2game2outcome[stim]['fun'].append(loser_fun)
            fun2game2outcome[stim]['outcome'].append('Lost')
        ended_early = accepted or surrender_info is not None
        if not ended_early:
            human_game_lens[stim].append(game_len)

In [ ]:
with open('../human-data/watch-exp/final_res.json', 'r') as f:
    watch_data = json.load(f)

all_games = sorted({game for user_data in watch_data.values() for game in user_data})

user2near_standard = {}
for user, user_data in watch_data.items():
    for game, game_data in user_data.items():
        judgements, payoff = game_data['judgements']
        judgements = judgements.values()
        near_standards = []
        for resp in judgements:
            near_standard = False
            if (resp > 45 and resp < 55) or (resp >= 0 and resp < 10) or (resp > 90):
                near_standard = True
            near_standards.append(near_standard)
        user2near_standard.setdefault(user, [])
        user2near_standard[user].extend(near_standards)

human_watch_payoffs = {game: [] for game in all_games}
human_watch_fun = {game: [] for game in all_games}
skipped_uid_watch = set()
for user, user_data in watch_data.items():
    for game, game_data in user_data.items():
        standard_judgements = user2near_standard[user]
        prop_standard = np.sum(standard_judgements)/len(standard_judgements)
        if prop_standard > 0.8:
            skipped_uid_watch.add(user)
            continue
        _, payoff = game_data['judgements']
        if payoff is not None:
            human_watch_payoffs[game].append(payoff)
        else:
            if 'response' in game_data['judgements'][0]:
                human_watch_fun[game].append(game_data['judgements'][0]['response'])

new_human_watch_fun = {}
new_human_watch_payoffs = {}
for game in human_watch_payoffs:
    task_vals = {'fun': human_watch_fun[game], 'advantage': human_watch_payoffs[game]}
    for task, resps in task_vals.items():
        n_resps = len(resps)
        n_rem = int(n_resps * 0.1)
        dists = []
        for idx in range(n_resps):
            resp = resps[idx]
            other_resps = resps[:idx] + resps[idx+1:]
            m = np.mean(other_resps)
            d = np.abs(m - resp)
            dists.append((idx, d))
        sorted_dists = sorted(dists, key=lambda x: x[1], reverse=True)
        rem = [x[0] for x in sorted_dists[:n_rem]]
        new_resps = [r for i, r in enumerate(resps) if i not in rem]
        if task == 'fun':
            new_human_watch_fun[game] = new_resps
        else:
            new_human_watch_payoffs[game] = new_resps
human_watch_fun = new_human_watch_fun
human_watch_payoffs = new_human_watch_payoffs
print('filtered watch uids:', len(skipped_uid_watch))

In [ ]:
with open('final_processed_res/think/human_processed.json.json', 'r') as f:
    think_data = json.load(f)

human_think_payoff = {game: vals['payoff'] for game, vals in think_data['human_payoff'].items()}
human_think_fun = {game: vals for game, vals in think_data['human_fun'].items()}

In [ ]:
tasks = ['think', 'play', 'watch']
queries = ['payoff', 'fun']
human_payoff = {'think': human_think_payoff,
                'play': human_play_payoffs,
                'watch': human_watch_payoffs}
human_fun = {'think': human_think_fun,
             'play': human_play_fun,
             'watch': human_watch_fun}
human_queries = {'fun': human_fun, 'payoff': human_payoff}

In [ ]:
random.seed(7)
np.random.seed(7)
n_bootstrap = 1000

for query in queries:
    h_queries = human_queries[query]
    for task in tasks:
        human_score_data = h_queries[task]
        ordered_games = sorted(list(human_score_data.keys()))
        r2_samples = []
        for _ in range(n_bootstrap):
            human_s1 = []
            human_s2 = []
            for game in ordered_games:
                h_scores = human_score_data[game]
                random.shuffle(h_scores)
                s1, s2 = np.array_split(h_scores, 2)
                human_s1.append(np.mean(s1))
                human_s2.append(np.mean(s2))
            r2 = stats.pearsonr(human_s1, human_s2)[0] ** 2
            r2_samples.append(r2)
        r2 = np.mean(r2_samples)
        r2_lower, r2_upper = analysis_utils.compute_ci(r2_samples)
        print(task, query, r2, r2_lower, r2_upper)

In [ ]:
random.seed(7)
np.random.seed(7)
subset_games = sorted(human_play_fun.keys())
r2_samples = []
for _ in range(n_bootstrap):
    s1, s2 = [], []
    for game in subset_games:
        h_scores = list(human_think_fun[game])
        random.shuffle(h_scores)
        a, b = np.array_split(h_scores, 2)
        s1.append(np.mean(a))
        s2.append(np.mean(b))
    r2_samples.append(stats.pearsonr(s1, s2)[0] ** 2)
r2 = np.mean(r2_samples)
r2_lower, r2_upper = analysis_utils.compute_ci(r2_samples)
print('think fun (41-game play subset)', r2, r2_lower, r2_upper)

In [ ]:
np.random.seed(7)
pairs = list(itertools.combinations(tasks, 2))
type2viz = {'watch': 'Post-Watch', 'think': 'Just Think', 'play': 'Post-Play'}
query2color = {'payoff': 'grey', 'fun': 'grey'}

for query in queries:
    for combo in pairs:
        s1, s2 = combo
        if 'watch' in str(combo):
            subgames = list(sorted(human_queries[query]['watch'].keys()))
        elif 'play' in str(combo):
            subgames = list(sorted(human_queries[query]['play'].keys()))

        src1_vals = human_queries[query][s1]
        src2_vals = human_queries[query][s2]
        m1 = [np.mean(src1_vals[game]) for game in subgames]
        m2 = [np.mean(src2_vals[game]) for game in subgames]

        boot_m1 = [[] for _ in subgames]
        boot_m2 = [[] for _ in subgames]
        r2_samples = []
        for _ in range(n_bootstrap):
            b1 = []
            b2 = []
            for i, game in enumerate(subgames):
                resample1 = np.random.choice(src1_vals[game], len(src1_vals[game]), replace=True)
                resample2 = np.random.choice(src2_vals[game], len(src2_vals[game]), replace=True)
                mean1 = np.mean(resample1)
                mean2 = np.mean(resample2)
                b1.append(mean1)
                b2.append(mean2)
                boot_m1[i].append(mean1)
                boot_m2[i].append(mean2)
            r2_samples.append(pearsonr(b1, b2)[0] ** 2)

        def get_ci(values):
            return np.percentile(values, 2.5), np.percentile(values, 97.5)

        m1_ci_low, m1_ci_high = [], []
        m2_ci_low, m2_ci_high = [], []
        for i in range(len(subgames)):
            low1, high1 = get_ci(boot_m1[i])
            low2, high2 = get_ci(boot_m2[i])
            m1_ci_low.append(m1[i] - low1)
            m1_ci_high.append(high1 - m1[i])
            m2_ci_low.append(m2[i] - low2)
            m2_ci_high.append(high2 - m2[i])

        mean_r2 = np.mean(r2_samples)
        r2_lower, r2_upper = np.percentile(r2_samples, [2.5, 97.5])

        fig, ax = plt.subplots()
        ax.scatter(m1, m2, alpha=0.7, color=query2color[query])
        ax.errorbar(m1, m2,
                    xerr=[m1_ci_low, m1_ci_high],
                    yerr=[m2_ci_low, m2_ci_high],
                    fmt='none', alpha=0.4, color=query2color[query])
        ax.set_xlabel(type2viz[s1], fontsize=20)
        ax.set_ylabel(type2viz[s2], fontsize=20)
        ax.set_title(f'{query}  R^2={mean_r2:.2f} [{r2_lower:.2f}, {r2_upper:.2f}]', fontsize=16)
        plt.tight_layout()
        plt.show()

In [ ]:
agent2features = {}
for agent in agents:
    if agent == 'ours':
        feature_df = pd.read_csv('../model-data/local_model_readout_fun_features.csv')
    elif agent == 'random':
        feature_df = pd.read_csv('../model-data/random_model_readout_fun_features.csv')
    else:
        feature_df = pd.read_csv('../model-data/expert_model_readout_fun_features.csv')
    agent2features[agent] = feature_df

model_model_lens = {agent: {} for agent in agents}
for agent in agents:
    fdf = agent2features[agent]
    for game in human_game_lens:
        model_model_lens[agent][game] = fdf.loc[fdf.game_id == game].iloc[0].len

In [ ]:
all_data = []
for game, data in fun2game2outcome.items():
    df = pd.DataFrame(data)
    all_data.append(df)
full_df = pd.concat(all_data, ignore_index=True)

outcomes = sorted(full_df['outcome'].unique())
hatch_patterns = ['/', '\\', 'x']
hatch_map = dict(zip(outcomes, hatch_patterns))

fig, axes = plt.subplots(nrows=3, figsize=(8, 10), sharex=True, sharey=True)
bins = range(0, 101, 10)
for ax, outcome in zip(axes, outcomes):
    subset = full_df[full_df['outcome'] == outcome]
    ax.hist(subset['fun'], bins=bins, color='gray', edgecolor='black', density=True,
            hatch=hatch_map[outcome])
    ax.set_title(f'{outcome}', fontsize=26)
    ax.set_ylabel('Density', fontsize=24)
axes[-1].set_xlabel('Funness', fontsize=24)
plt.tight_layout()
plt.show()

In [ ]:
low_emd_data = []
high_emd_data = []
valid_games = set(fun2game2outcome.keys())
filtered_feature_df = feature_df[feature_df['game_id'].isin(valid_games)]
emd_values = filtered_feature_df.set_index('game_id')['emd']
emd_25 = np.percentile(emd_values, 25)
emd_75 = np.percentile(emd_values, 75)

for game, fun_res in fun2game2outcome.items():
    emd = emd_values.get(game, np.nan)
    if pd.isna(emd):
        continue
    for fun_val, outcome in zip(fun_res['fun'], fun_res['outcome']):
        row = {'fun': fun_val, 'outcome': outcome}
        if emd < emd_25:
            low_emd_data.append(row)
        elif emd > emd_75:
            high_emd_data.append(row)

def plot_funness_distributions(data, title, axarr):
    outcomes = ['Draw', 'Lost', 'Won']
    hatch_patterns = ['/', '\\', 'x']
    hatch_map = dict(zip(outcomes, hatch_patterns))
    bins = range(0, 101, 10)
    for ax, outcome in zip(axarr, outcomes):
        subset = pd.DataFrame(data)
        subset = subset[subset['outcome'] == outcome]
        ax.hist(subset['fun'], bins=bins, color='gray', edgecolor='black', density=True,
                hatch=hatch_map[outcome])
        ax.set_title(f'{outcome}', fontsize=24)
        ax.set_ylabel('Density', fontsize=24)
    axarr[-1].set_xlabel('Funness', fontsize=24)

fig, axes = plt.subplots(nrows=3, figsize=(8, 10), sharex=True, sharey=True)
plot_funness_distributions(low_emd_data, 'Unbalanced', axes)
plt.show()

fig, axes = plt.subplots(nrows=3, figsize=(8, 10), sharex=True, sharey=True)
plot_funness_distributions(high_emd_data, 'Balanced', axes)
plt.show()

In [ ]:
subgames = sorted(human_game_lens.keys())
fig, axes = plt.subplots(1, len(agents), figsize=(10, 4), sharex=True, sharey=True)
for i, agent in enumerate(agents):
    ax = axes[i]
    m_lens = [model_model_lens[agent][game] for game in subgames]
    h_lens = [np.mean(human_game_lens[game]) for game in subgames]
    h_len_std = [np.std(human_game_lens[game]) for game in subgames]
    ax.scatter(m_lens, h_lens, color=agent2color[agent], alpha=0.5)
    ax.errorbar(m_lens, h_lens, yerr=h_len_std, fmt='.', alpha=0.2, color=agent2color[agent])
    lb = 0
    ub = 60
    ax.plot([lb, ub], [lb, ub], 'k--', alpha=0.7)
    r_val, _ = pearsonr(m_lens, h_lens)
    r2 = r_val ** 2
    ax.text(0.05, 0.85, f'$R^2 = {r2:.2f}$', transform=ax.transAxes, fontsize=20)
    ax.set_xlabel('Predicted Length', fontsize=16)
    if i == 0:
        ax.set_ylabel('Empirical Length', fontsize=16)
    ax.set_title(model2name[agent], fontsize=24)
fig.tight_layout()
plt.savefig('final_figures/play_think_length.pdf', dpi=400)

In [ ]:
subgames = sorted(game2draws.keys())
payoffs = human_queries['payoff']['think']
preds = [np.abs(0 - np.mean(payoffs[game])) for game in subgames]
n_occ = [len(game2draws[game])/game2n_matches[game] * 100 for game in subgames]
fig, ax = plt.subplots()
ax.scatter(preds, n_occ, color=human_color, s=100, alpha=0.7)
ax.set_xlabel('Predicted Payoff Bias', fontsize=24)
ax.set_ylabel('% Draw Request', fontsize=24)
print(pearsonr(preds, n_occ))
print(pearsonr(preds, n_occ)[0]**2)
ax.set_title('People', fontsize=24)
ax.set_xlim([-0.1, 1.1])
plt.savefig('final_figures/draw_vs_payoff.png', dpi=400)

In [ ]:
subgames = sorted(game2draws.keys())
payoffs = human_queries['payoff']['think']
preds = [np.abs(0 - np.mean(payoffs[game])) for game in subgames]
n_occ = [game2surrenders[game]/game2n_matches[game] * 100 for game in subgames]
fig, ax = plt.subplots()
ax.scatter(preds, n_occ, s=100, alpha=0.7, color=human_color)
ax.set_xlabel('Predicted Payoff Bias', fontsize=24)
ax.set_ylabel('% Surrender', fontsize=24)
print(pearsonr(preds, n_occ))
print(pearsonr(preds, n_occ)[0]**2)
ax.set_title('People', fontsize=24)
ax.set_xlim([-0.1, 1.1])
plt.savefig('final_figures/surr_vs_payoff.png', dpi=400)

In [ ]:
subgames = sorted(game2draws.keys())
fun_vals = human_queries['fun']['think']
preds = [np.abs(0 - np.mean(fun_vals[game])) for game in subgames]
n_occ = [len(game2draws[game])/game2n_matches[game] * 100 for game in subgames]
fig, ax = plt.subplots()
ax.scatter(preds, n_occ, s=100, alpha=0.7, color=human_color)
ax.set_xlabel('Predicted Fun', fontsize=24)
ax.set_ylabel('% Draw Request', fontsize=24)
print(pearsonr(preds, n_occ))
print(pearsonr(preds, n_occ)[0]**2)
ax.set_title('People', fontsize=24)
ax.set_xlim([0, 80])
plt.savefig('final_figures/draw_v_fun.png', dpi=400)

In [ ]:
subgames = sorted(game2draws.keys())
fun_vals = human_queries['fun']['think']
preds = [np.abs(0 - np.mean(fun_vals[game])) for game in subgames]
n_occ = [game2surrenders[game]/game2n_matches[game] * 100 for game in subgames]
fig, ax = plt.subplots()
ax.scatter(preds, n_occ, s=100, alpha=0.7, color=human_color)
ax.set_xlabel('Predicted Fun', fontsize=24)
ax.set_ylabel('% Surrender', fontsize=24)
print(pearsonr(preds, n_occ))
print(pearsonr(preds, n_occ)[0]**2)
ax.set_title('People', fontsize=24)
ax.set_xlim([0, 80])
plt.savefig('final_figures/surrender_v_fun.png', dpi=400)